# FlashRank-Pro Training Pipeline
Each cell runs one stage. All output writes directly to Google Drive.
Session breaks are safe — just re-run all, it skips completed stages.

In [ ]:
# SETUP
import os, shutil, subprocess, time

DRIVE = "/content/drive/MyDrive/flashrank-pro"
LOCAL = "/content/flashrank-pro"

# Install
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers sentence-transformers accelerate datasets fire tqdm peft openai huggingface-hub torchao>=0.16.0

# Mount Drive
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive", force_remount=False)
os.makedirs(f"{DRIVE}/data", exist_ok=True)
os.makedirs(f"{DRIVE}/models", exist_ok=True)

# Get code — verify actual training file exists
CHECKFILE = f"{LOCAL}/training/01_generate_synthetic_data.py"
if not os.path.exists(CHECKFILE):
    if os.path.exists(LOCAL):
        shutil.rmtree(LOCAL)
    got = False
    try:
        import requests, zipfile, io
        url = "https://api.github.com/repos/eulogik/flashrank-pro/zipball/main"
        headers = {}
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")
            if token: headers["Authorization"] = f"Bearer {token.strip()}"
        except Exception:
            pass
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()
        z = zipfile.ZipFile(io.BytesIO(resp.content))
        _tmp = LOCAL + "-tmp"
        z.extractall(_tmp)
        os.rename(os.path.join(_tmp, os.listdir(_tmp)[0]), LOCAL)
        shutil.rmtree(_tmp, ignore_errors=True)
        got = True
    except Exception as e:
        print(f"Zipball failed: {e}")
    if not got:
        result = subprocess.run(
            ["git", "clone", "https://github.com/eulogik/flashrank-pro.git", LOCAL],
            capture_output=True, text=True, timeout=60)
        if result.returncode == 0:
            got = True
        else:
            print(f"git clone failed: {result.stderr.strip()}")
    if not os.path.exists(CHECKFILE):
        raise RuntimeError(
            "Code download FAILED. Do one of:\n"
            "  1. Set GH_TOKEN in Colab secrets (key sidebar), then re-run setup\n"
            "  2. Run manually: !rm -rf /content/flashrank-pro && git clone https://github.com/eulogik/flashrank-pro.git /content/flashrank-pro\n"
            "  3. Check your internet connection")
    print("Got code")

# Restore from Drive
for d in ["data", "models"]:
    src, dst = f"{DRIVE}/{d}", f"{LOCAL}/{d}"
    if os.path.isdir(src) and os.listdir(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Restored {d}/")
    else:
        os.makedirs(dst, exist_ok=True)

# GPU
import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory/1e9:.0f}GB)")

# Status
entries = [
    ("data",   f"{DRIVE}/data/synthetic_training_data.jsonl"),
    ("kd",     f"{DRIVE}/models/flashrank-pro-base-kd-en"),
    ("rl",     f"{DRIVE}/models/flashrank-pro-base-rl"),
    ("merged", f"{DRIVE}/models/flashrank-pro-base-merged"),
]
for name, path in entries:
    print(f"  {'ok' if os.path.exists(path) else 'pending'}: {name}")

## Stage 1: Generate Training Data (~2.5h)
Writes to Drive directly. Checkpoints survive session breaks.

In [ ]:
DATA_FILE = f"/content/drive/MyDrive/flashrank-pro/data/synthetic_training_data.jsonl"
LOCAL_DATA = f"/content/flashrank-pro/data/synthetic_training_data.jsonl"

os.chdir("/content/flashrank-pro")
!python training/01_generate_synthetic_data.py \
    --output_path {DATA_FILE} \
    --corpus_name sentence-transformers/gooaq \
    --n_queries 50000 \
    --n_negatives 4

if os.path.exists(DATA_FILE):
    os.makedirs(os.path.dirname(LOCAL_DATA), exist_ok=True)
    shutil.copy2(DATA_FILE, LOCAL_DATA)
    lines = sum(1 for _ in open(DATA_FILE))
    print(f"Done: {lines} examples")
else:
    print("Output not found \u2014 check logs above")

## Stage 2: Knowledge Distillation (~3h on T4)

In [ ]:
SIZE = "base"
OUT = f"/content/drive/MyDrive/flashrank-pro/models/flashrank-pro-{SIZE}-kd-en"
DATA = f"/content/drive/MyDrive/flashrank-pro/data/synthetic_training_data.jsonl"

os.chdir("/content/flashrank-pro")
if os.path.exists(OUT):
    print(f"Already done: {OUT}")
elif not os.path.exists(DATA):
    print(f"Data not found: {DATA}")
else:
    !python training/02_knowledge_distillation.py \
        --model_name answerdotai/ModernBERT-{SIZE} \
        --data_path {DATA} \
        --output_dir {OUT} \
        --batch_size 8 \
        --num_epochs 3 \
        --learning_rate 2e-5

## Stage 3: GRPO Reinforcement Learning (~1-2h)
Uses LoRA. Requires Stage 2 checkpoint on Drive.

In [ ]:
SIZE = "base"
IN  = f"/content/drive/MyDrive/flashrank-pro/models/flashrank-pro-{SIZE}-kd-en"
OUT = f"/content/drive/MyDrive/flashrank-pro/models/flashrank-pro-{SIZE}-rl"
DATA = f"/content/drive/MyDrive/flashrank-pro/data/synthetic_training_data.jsonl"

os.chdir("/content/flashrank-pro")
if os.path.exists(OUT):
    print(f"Already done: {OUT}")
elif not os.path.exists(IN):
    print(f"Stage 2 input not found: {IN}")
elif not os.path.exists(DATA):
    print(f"Data not found: {DATA}")
else:
    !python training/03_grpo_rl.py \
        --model_path {IN} \
        --data_path {DATA} \
        --output_dir {OUT} \
        --batch_size 4 \
        --num_epochs 1

## Stage 4: SLERP Merge (~5min)
Merges KD + RL checkpoints on Drive.

In [ ]:
import json
SIZE = "base"
KD  = f"/content/drive/MyDrive/flashrank-pro/models/flashrank-pro-{SIZE}-kd-en"
RL  = f"/content/drive/MyDrive/flashrank-pro/models/flashrank-pro-{SIZE}-rl"
OUT = f"/content/drive/MyDrive/flashrank-pro/models/flashrank-pro-{SIZE}-merged"

os.chdir("/content/flashrank-pro")
if os.path.exists(OUT):
    print("Already done")
else:
    available = [p for p in [KD, RL] if os.path.exists(p)]
    if len(available) < 2:
        print(f"Need 2 checkpoints, found {len(available)}")
    else:
        cfg = {"checkpoints": available, "weights": [0.5, 0.5]}
        os.makedirs("configs", exist_ok=True)
        with open("configs/slerp_config.json", "w") as f:
            json.dump(cfg, f)
        !python training/04_slerp_merge.py \
            --config_path configs/slerp_config.json \
            --output_path {OUT}

## Sanity Check

In [ ]:
SIZE = "base"
MERGED = f"/content/drive/MyDrive/flashrank-pro/models/flashrank-pro-{SIZE}-merged"

os.chdir("/content/flashrank-pro")
if os.path.exists(MERGED):
    from flashrank_pro import Reranker
    r = Reranker(MERGED, device="cuda" if __import__('torch').cuda.is_available() else "cpu")
    results = r.rerank("how to train a neural network", [
        "Training neural networks requires backpropagation.",
        "Python is a programming language.",
        "Gradient descent optimizes loss functions."])
    for i, res in enumerate(results, 1):
        print(f"{i}. [{res['score']:.4f}] {res['text'][:60]}")
else:
    print("No merged model yet")

## Deploy to HuggingFace

In [ ]:
SIZE = "base"
MERGED = f"/content/drive/MyDrive/flashrank-pro/models/flashrank-pro-{SIZE}-merged"

os.chdir("/content/flashrank-pro")
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token and os.path.exists(MERGED):
        !python scripts/deploy_to_huggingface.py --model_path {MERGED} --repo_id eulogik/flashrank-pro-{SIZE}
except Exception:
    print("No HF_TOKEN or no model")